In [0]:
%sql
ALTER TABLE adbdevbankproject.silver.transactions_data_external
RENAME TO adbdevbankproject.silver.transactions_data;

In [0]:
%sql
SELECT COUNT(*) AS row_count
FROM adbdevbankproject.silver.transactions_data;

DESCRIBE adbdevbankproject.silver.transactions_data;

col_name,data_type,comment
id,int,null
client_id,int,null
card_id,int,null
transaction_date,date,null
transaction_time,string,null
amount,"decimal(18,2)",null
use_chip,string,null
merchant_id,int,null
merchant_city,string,null
merchant_state,string,null


In [0]:
%sql
DESCRIBE DETAIL adbdevbankproject.silver.transactions_data_external;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,dc3eb1da-0843-4252-a182-d7b036f008d2,adbdevbankproject.silver.transactions_data_external,null,abfss://silver@stgdevbankproject.dfs.core.windows.net/transactions_data,2026-08-21T12:54:01.661Z,2026-08-21T12:54:16.000Z,List(),List(),3,230140446,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
-- Write the completed Silver transactions table to the Silver ADLS container

CREATE TABLE adbdevbankproject.silver.transactions_data_external
USING DELTA
LOCATION 'abfss://silver@stgdevbankproject.dfs.core.windows.net/transactions_data'
AS
SELECT *
FROM adbdevbankproject.silver.transactions_data;

num_affected_rows,num_inserted_rows


In [0]:
%sql

DESCRIBE DETAIL adbdevbankproject.silver.transactions_data;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,eef4f66c-37d4-4e5a-9464-70caf6a8ce1e,adbdevbankproject.silver.transactions_data,null,,2026-08-18T11:41:50.384Z,2026-08-18T12:00:25.000Z,List(),List(),4,162950704,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 286525, numDeletionVectors -> 3)",false


In [0]:
%sql

-- Correct merchant_country for transactions where merchant_state is Washington (WA)

UPDATE adbdevbankproject.silver.transactions_data
SET merchant_country = 'United States'
WHERE merchant_state = 'WA';



-- Validate the corrected Washington geographic mapping

SELECT
    merchant_country,
    merchant_state,
    COUNT(*) AS transaction_count
FROM adbdevbankproject.silver.transactions_data
WHERE merchant_state = 'WA'
GROUP BY
    merchant_country,
    merchant_state;

merchant_country,merchant_state,transaction_count
United States,WA,286525


In [0]:
%sql

-- Create the cleaned Transactions Silver table
-- Apply all approved data quality and transformation rules

CREATE OR REPLACE TABLE adbdevbankproject.silver.transactions_data
USING DELTA
AS

SELECT

    -- Transaction identifiers
    id,
    client_id,
    card_id,

    -- Split transaction timestamp into separate date and time columns
    CAST(date AS DATE) AS transaction_date,
    DATE_FORMAT(date, 'HH:mm:ss') AS transaction_time,

    -- Convert amount from currency string to decimal
    CAST(
        REPLACE(amount, '$', '') AS DECIMAL(18,2)
    ) AS amount,

    -- Transaction method
    use_chip,

    -- Merchant information
    merchant_id,

    -- Convert ONLINE to NULL because it is not a physical city
    CASE
        WHEN merchant_city = 'ONLINE' THEN NULL
        ELSE merchant_city
    END AS merchant_city,

    -- Keep only valid US state codes
    CASE
        WHEN merchant_state IN (
            'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA',
            'HI','ID','IL','IN','IA','KS','KY','LA','ME','MD',
            'MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ',
            'NM','NY','NC','ND','OH','OK','OR','PA','RI','SC',
            'SD','TN','TX','UT','VT','VA','WA','WV','WI','WY','DC'
        )
        THEN merchant_state
        ELSE NULL
    END AS merchant_state,

    -- Assign United States to US state transactions
    -- Keep country names for non-US transactions
    CASE
        WHEN merchant_state IN (
            'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA',
            'HI','ID','IL','IN','IA','KS','KY','LA','ME','MD',
            'MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ',
            'NM','NY','NC','ND','OH','OK','OR','PA','RI','SC',
            'SD','TN','TX','UT','VT','VA','WV','WI','WY','DC'
        )
        THEN 'United States'

        WHEN merchant_state IS NULL OR merchant_state = 'AA'
        THEN NULL

        ELSE merchant_state
    END AS merchant_country,

    -- Convert ZIP code from DOUBLE to STRING
    CAST(CAST(zip AS BIGINT) AS STRING) AS zip,

    -- Merchant Category Code
    mcc,

    -- Replace NULL errors with an explicit no-error value
    COALESCE(errors, 'No Error') AS errors

FROM adbdevbankproject.bronze.transactions_data

-- Exclude the 6 transactions with invalid merchant_state = 'AA'
WHERE merchant_state <> 'AA'
   OR merchant_state IS NULL;

num_affected_rows,num_inserted_rows


In [0]:
%sql

-- Check the errors column for NULL values and count the distinct error types

SELECT
    COUNT(*) AS total_rows,
    COUNT(errors) AS non_null_errors,
    COUNT(*) - COUNT(errors) AS null_errors,
    COUNT(DISTINCT errors) AS unique_errors
FROM adbdevbankproject.bronze.transactions_data;


-- Review the distinct error values and their frequency

SELECT
    errors,
    COUNT(*) AS transaction_count
FROM adbdevbankproject.bronze.transactions_data
GROUP BY errors
ORDER BY transaction_count DESC;



errors,transaction_count
null,13094522
Insufficient Balance,130902
Bad PIN,32119
Technical Glitch,26271
Bad Card Number,7767
Bad Expiration,6161
Bad CVV,6106
Bad Zipcode,1126
"Bad PIN,Insufficient Balance",293
"Insufficient Balance,Technical Glitch",243


In [0]:
%sql

-- Check the mcc column for NULL values, unique MCC codes, and invalid negative values

SELECT
    COUNT(*) AS total_rows,
    COUNT(mcc) AS non_null_mcc,
    COUNT(*) - COUNT(mcc) AS null_mcc,
    COUNT(DISTINCT mcc) AS unique_mcc,
    SUM(CASE WHEN mcc < 0 THEN 1 ELSE 0 END) AS negative_mcc
FROM adbdevbankproject.bronze.transactions_data;




-- Review the minimum and maximum MCC values

SELECT
    MIN(mcc) AS min_mcc,
    MAX(mcc) AS max_mcc
FROM adbdevbankproject.bronze.transactions_data;

min_mcc,max_mcc
1711,9402


In [0]:
%sql

-- Check the zip column for NULL values, unique ZIP codes, and negative values

SELECT
    COUNT(*) AS total_rows,
    COUNT(zip) AS non_null_zips,
    COUNT(*) - COUNT(zip) AS null_zips,
    COUNT(DISTINCT zip) AS unique_zips,
    SUM(CASE WHEN zip < 0 THEN 1 ELSE 0 END) AS negative_zips
FROM adbdevbankproject.bronze.transactions_data;



-- Preview ZIP code values and their current DOUBLE representation

SELECT DISTINCT
    zip
FROM adbdevbankproject.bronze.transactions_data
WHERE zip IS NOT NULL
ORDER BY zip
LIMIT 30;



-- Check the minimum and maximum ZIP code length after converting from DOUBLE

SELECT
    MIN(LENGTH(CAST(CAST(zip AS BIGINT) AS STRING))) AS min_zip_length,
    MAX(LENGTH(CAST(CAST(zip AS BIGINT) AS STRING))) AS max_zip_length
FROM adbdevbankproject.bronze.transactions_data
WHERE zip IS NOT NULL;




-- Convert ZIP codes from DOUBLE to STRING while preserving their original values

SELECT
    CAST(CAST(zip AS BIGINT) AS STRING) AS zip
FROM adbdevbankproject.bronze.transactions_data;

zip
35601
78155
60510
11717
68641
55116
12302
98516
71730
10025


In [0]:
%sql
-- Check the merchant_state column for NULL values and the number of unique states

SELECT
    COUNT(*) AS total_rows,
    COUNT(merchant_state) AS non_null_states,
    COUNT(*) - COUNT(merchant_state) AS null_states,
    COUNT(DISTINCT merchant_state) AS unique_states
FROM adbdevbankproject.bronze.transactions_data;


-- Review the distinct merchant state values to identify missing, invalid, or inconsistent values

SELECT
    merchant_state,
    COUNT(*) AS transaction_count
FROM adbdevbankproject.bronze.transactions_data
GROUP BY merchant_state
ORDER BY transaction_count DESC;


-- Investigate the transactions with the unusual merchant_state value 'AA'

SELECT
    id,
    merchant_city,
    merchant_state,
    zip,
    merchant_id,
    amount
FROM adbdevbankproject.bronze.transactions_data
WHERE merchant_state = 'AA';
/*
القرار النهائي هو:

merchant_city يبقى كما هو.
merchant_state يبقى فقط للـ US states.
merchant_country يكون United States للـ US states.
الدول الأخرى تنتقل إلى merchant_country.
AA كل الترانزاكشن اللي مرتبط ب ستيت تحذف لعدم وجود ولاية بهذا الاسم او الاختصار
*/



-- Investigate transactions where merchant_city is recorded as 'Online'

SELECT
    merchant_city,
    merchant_state,
    use_chip,
    COUNT(*) AS transaction_count
FROM adbdevbankproject.bronze.transactions_data
WHERE merchant_city = 'ONLINE'
GROUP BY
    merchant_city,
    merchant_state,
    use_chip
ORDER BY transaction_count DESC;




-- Count transactions where merchant_city is recorded as ONLINE

SELECT
    COUNT(*) AS online_city_transactions
FROM adbdevbankproject.bronze.transactions_data
WHERE merchant_city = 'ONLINE';




online_city_transactions
1563700


In [0]:
%sql
-- Check the merchant_city column for NULL values and the number of unique cities

SELECT
    COUNT(*) AS total_rows,
    COUNT(merchant_city) AS non_null_cities,
    COUNT(*) - COUNT(merchant_city) AS null_cities,
    COUNT(DISTINCT merchant_city) AS unique_cities
FROM adbdevbankproject.bronze.transactions_data;


-- Review sample merchant city values to identify formatting or data quality issues

SELECT DISTINCT merchant_city
FROM adbdevbankproject.bronze.transactions_data
LIMIT 30;


-- Check for whitespace in merchant city names

SELECT
    COUNT(*) AS rows_with_extra_spaces
FROM adbdevbankproject.bronze.transactions_data
WHERE merchant_city <> TRIM(merchant_city);


-- قرار Silver لـ merchant_city
--NULL = 0
--لا توجد مسافات زائدة
--القيم تبدو صحيحة
-- سترنق نوع مناسب



rows_with_extra_spaces
0


In [0]:
%sql
-- Check the merchant_id column for NULL values, unique merchants, and invalid negative values

SELECT
    COUNT(*) AS total_rows,
    COUNT(merchant_id) AS non_null_merchant_ids,
    COUNT(*) - COUNT(merchant_id) AS null_merchant_ids,
    COUNT(DISTINCT merchant_id) AS unique_merchant_ids,
    SUM(CASE WHEN merchant_id < 0 THEN 1 ELSE 0 END) AS negative_merchant_ids
FROM adbdevbankproject.bronze.transactions_data;

total_rows,non_null_merchant_ids,null_merchant_ids,unique_merchant_ids,negative_merchant_ids
13305915,13305915,0,74831,0


In [0]:
%sql
-- Check the distinct values and frequency of the use_chip column
-- كل القيم الموجودة وعدد كل قيمة

SELECT
    use_chip,
    COUNT(*) AS transaction_count
FROM adbdevbankproject.bronze.transactions_data
GROUP BY use_chip
ORDER BY transaction_count DESC;

-- النتيجة:  
-- لا يحتاج تنظيف جوهري. القيم واضحة وموحدة، وما عندنا مثلًا اختلافات من نوع:
--Chip
--chip transaction
--CHIP Transaction


use_chip,transaction_count
Swipe Transaction,6967185
Chip Transaction,4780818
Online Transaction,1557912


In [0]:
%sql
-- Check the amount column for NULL values and review sample values

SELECT
    COUNT(*) AS total_rows,
    COUNT(amount) AS non_null_amounts,
    COUNT(*) - COUNT(amount) AS null_amounts
FROM adbdevbankproject.bronze.transactions_data;

-- Preview sample values from the amount column before cleaning

SELECT DISTINCT amount
FROM adbdevbankproject.bronze.transactions_data
LIMIT 20;

-- Check whether all amount values follow the expected dollar format

SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN amount LIKE '$%' THEN 1 ELSE 0 END) AS dollar_format,
    SUM(CASE WHEN amount NOT LIKE '$%' THEN 1 ELSE 0 END) AS other_format
FROM adbdevbankproject.bronze.transactions_data;


-- Clean the amount column by removing the dollar sign and converting it to DECIMAL

SELECT
    id,
    amount AS original_amount,
    CAST(REPLACE(amount, '$', '') AS DECIMAL(10,2)) AS amount
FROM adbdevbankproject.bronze.transactions_data
LIMIT 20;

-- النتيجة: تحويله الى ديسمل وجعل الفورمات $ من قبل المحلل

id,original_amount,amount
15549809,$80.00,80.00
15549810,$55.49,55.49
15549811,$50.98,50.98
15549813,$71.08,71.08
15549815,$26.96,26.96
15549816,$26.65,26.65
15549818,$20.21,20.21
15549819,$42.99,42.99
15549821,$27.63,27.63
15549822,$53.79,53.79


In [0]:
%sql
-- Check the card_id column for NULL values, duplicates, and invalid negative values

SELECT
    COUNT(*) AS total_rows,
    COUNT(card_id) AS non_null_card_ids,
    COUNT(*) - COUNT(card_id) AS null_card_ids,
    COUNT(DISTINCT card_id) AS unique_card_ids,
    SUM(CASE WHEN card_id < 0 THEN 1 ELSE 0 END) AS negative_card_ids
FROM adbdevbankproject.bronze.transactions_data;

total_rows,non_null_card_ids,null_card_ids,unique_card_ids,negative_card_ids
13305915,13305915,0,4071,0


In [0]:
%sql
-- Check the client_id column for NULL values, duplicates, and invalid negative values

SELECT
    COUNT(*) AS total_rows,
    COUNT(client_id) AS non_null_client_ids,
    COUNT(*) - COUNT(client_id) AS null_client_ids,
    COUNT(DISTINCT client_id) AS unique_client_ids,
    SUM(CASE WHEN client_id < 0 THEN 1 ELSE 0 END) AS negative_client_ids
FROM adbdevbankproject.bronze.transactions_data;

total_rows,non_null_client_ids,null_client_ids,unique_client_ids,negative_client_ids
13305915,13305915,0,1219,0


In [0]:
%sql
-- Check the date column for NULL values and the minimum and maximum transaction dates

SELECT
    COUNT(*) AS total_rows,
    COUNT(date) AS non_null_dates,
    COUNT(*) - COUNT(date) AS null_dates,
    MIN(date) AS min_date,
    MAX(date) AS max_date
FROM adbdevbankproject.bronze.transactions_data;

-- Preview the date column to verify the timestamp format and values

SELECT
    date
FROM adbdevbankproject.bronze.transactions_data
LIMIT 20;

-- Extract the transaction date and time into separate columns

SELECT
    id,
    date,
    CAST(date AS DATE) AS transaction_date,
    DATE_FORMAT(date, 'HH:mm:ss') AS transaction_time
FROM adbdevbankproject.bronze.transactions_data
LIMIT 20;

-- النتيجة: فصله الى عاموين 

id,date,transaction_date,transaction_time
15549809,2015-01-17T21:30:00.000Z,2015-01-17,21:30:00
15549810,2015-01-17T21:30:00.000Z,2015-01-17,21:30:00
15549811,2015-01-17T21:30:00.000Z,2015-01-17,21:30:00
15549813,2015-01-17T21:31:00.000Z,2015-01-17,21:31:00
15549815,2015-01-17T21:32:00.000Z,2015-01-17,21:32:00
15549816,2015-01-17T21:32:00.000Z,2015-01-17,21:32:00
15549818,2015-01-17T21:33:00.000Z,2015-01-17,21:33:00
15549819,2015-01-17T21:33:00.000Z,2015-01-17,21:33:00
15549821,2015-01-17T21:34:00.000Z,2015-01-17,21:34:00
15549822,2015-01-17T21:34:00.000Z,2015-01-17,21:34:00


In [0]:
%sql
-- Check the ID column for NULL values, duplicate IDs, and invalid negative values

SELECT
    COUNT(*) AS total_rows,
    COUNT(id) AS non_null_ids,
    COUNT(*) - COUNT(id) AS null_ids,
    COUNT(DISTINCT id) AS unique_ids,
    SUM(CASE WHEN id < 0 THEN 1 ELSE 0 END) AS negative_ids
FROM adbdevbankproject.bronze.transactions_data;

total_rows,non_null_ids,null_ids,unique_ids,negative_ids
13305915,13305915,0,13305915,0


In [0]:
%sql
-- Read the Bronze transactions table and preview the data

SELECT *
FROM adbdevbankproject.bronze.transactions_data
LIMIT 20;

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
15549809,2015-01-17T21:30:00.000Z,1046,9,$80.00,Chip Transaction,27092,Philadelphia,PA,19145.0,4829,null
15549810,2015-01-17T21:30:00.000Z,1240,3876,$55.49,Online Transaction,23446,ONLINE,null,null,7801,null
15549811,2015-01-17T21:30:00.000Z,1543,3864,$50.98,Online Transaction,15143,ONLINE,null,null,4784,null
15549813,2015-01-17T21:31:00.000Z,1897,4948,$71.08,Swipe Transaction,38339,Huntington,NY,11743.0,5813,null
15549815,2015-01-17T21:32:00.000Z,1541,2307,$26.96,Chip Transaction,81223,Clinton,TN,37716.0,4121,null
15549816,2015-01-17T21:32:00.000Z,1737,6007,$26.65,Chip Transaction,52433,Eastpointe,MI,48021.0,5813,null
15549818,2015-01-17T21:33:00.000Z,542,4195,$20.21,Chip Transaction,99130,Texarkana,TX,75501.0,5813,null
15549819,2015-01-17T21:33:00.000Z,1095,5461,$42.99,Chip Transaction,19423,Ridgewood,NY,11385.0,5812,null
15549821,2015-01-17T21:34:00.000Z,457,3129,$27.63,Chip Transaction,55060,Charlotte,NC,28215.0,5812,null
15549822,2015-01-17T21:34:00.000Z,860,5782,$53.79,Chip Transaction,44578,Silver Spring,MD,20901.0,5812,null
